# AVISO Eddy Dataset Pipeline

This notebook is a concise control panel for the AVISO surface eddy pipeline. Scientific and orchestration code lives in `src/aviso_eddy_dataset/`; this notebook loads configuration, runs stages, and checks outputs.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

AVISO_SRC = PROJECT_ROOT / "src"
SEACOFS_SRC = PROJECT_ROOT.parent / "seacofs_eddy_dataset_modular" / "src"
for source in (AVISO_SRC, SEACOFS_SRC):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

PROJECT_ROOT

PosixPath('/home/z5297792/UNSW-PhD/aviso_eddy_dataset')

## Load configuration

The notebook uses `config/local.yaml` when present and otherwise falls back to the committed `config/example.yaml`.

In [2]:
from aviso_eddy_dataset.config import load_config
from aviso_eddy_dataset.pipeline import STAGES, run_all, run_stage

LOCAL_CONFIG = PROJECT_ROOT / "config" / "local.yaml"
CONFIG_PATH = LOCAL_CONFIG if LOCAL_CONFIG.exists() else PROJECT_ROOT / "config" / "example.yaml"
config = load_config(CONFIG_PATH)

print(f"Config: {CONFIG_PATH}")
print(f"Input:  {config.data_root}")
print(f"Output: {config.output_root}")

Config: /home/z5297792/UNSW-PhD/aviso_eddy_dataset/config/local.yaml
Input:  /srv/scratch/z5502183/AVISO_0.125
Output: /srv/scratch/z5297792/aviso_eddy_dataset


In [3]:
[(stage.name, stage.description) for stage in STAGES]

[('detect_nencioli',
  'Detect daily candidates from 1 km interpolated ugos/vgos.'),
 ('fit_doppio_surface', 'Fit DOPPIO on native-resolution ugos/vgos.'),
 ('track_eddies',
  'Track surface eddies continuously across all source files.'),
 ('process_tracked_dataset',
  'Apply QC and create the processed AVISO dataset.')]

## Run one stage

Uncomment a stage while developing or resuming the workflow.

In [4]:
# run_stage("detect_nencioli", config)
# run_stage("fit_doppio_surface", config)
# run_stage("track_eddies", config)
# run_stage("process_tracked_dataset", config)

## Run selected stages

Uncomment the stages to run. Keep their pipeline order. With `skip_existing: true`, completed annual partitions are skipped.

In [5]:
STAGES_TO_RUN = [
    # "detect_nencioli",
    # "fit_doppio_surface",
    "track_eddies",
    "process_tracked_dataset",
]

for stage_name in STAGES_TO_RUN:
    run_stage(stage_name, config)

/srv/scratch/z5297792/aviso_eddy_dataset/tracked/eddy_tracks.parquet
/srv/scratch/z5297792/aviso_eddy_dataset/processed/eddy_dataset_processed.parquet


## Inspect outputs

In [6]:
import pandas as pd

output_dirs = {
    "detections": config.output_root / "detections",
    "surface_eddies": config.output_root / "surface_eddies",
    "tracked": config.output_root / "tracked",
    "processed": config.output_root / "processed",
}

pd.DataFrame(
    [
        {
            "stage": name,
            "path": str(path),
            "parquet_files": len(list(path.glob("*.parquet"))) if path.exists() else 0,
        }
        for name, path in output_dirs.items()
    ]
)

,stage,path,parquet_files
0,detections,/srv/scratch/z5297792/aviso_eddy_dataset/detec...,26
1,surface_eddies,/srv/scratch/z5297792/aviso_eddy_dataset/surfa...,26
2,tracked,/srv/scratch/z5297792/aviso_eddy_dataset/tracked,1
3,processed,/srv/scratch/z5297792/aviso_eddy_dataset/proce...,1


In [7]:
detection_files = sorted((config.output_root / "detections").glob("source=*.parquet"))

if detection_files:
    detection_counts = []
    for path in detection_files:
        frame = pd.read_parquet(path, columns=["Day", "Cyc"])
        detection_counts.append(
            {
                "partition": path.stem,
                "rows": len(frame),
                "first_day": frame.Day.min() if not frame.empty else pd.NA,
                "last_day": frame.Day.max() if not frame.empty else pd.NA,
                "AE": int(frame.Cyc.eq("AE").sum()),
                "CE": int(frame.Cyc.eq("CE").sum()),
            }
        )
    display(pd.DataFrame(detection_counts))
else:
    print(f"No detection files found in {config.output_root / 'detections'}")

,partition,rows,first_day,last_day,AE,CE
0,source=AVISO_0.125_EAC_1994,26312,16072,16435,12765,13547
1,source=AVISO_0.125_EAC_1995,28743,16436,16800,13479,15264
2,source=AVISO_0.125_EAC_1996,26198,16801,17166,11950,14248
3,source=AVISO_0.125_EAC_1997,29028,17167,17531,13699,15329
4,source=AVISO_0.125_EAC_1998,27337,17532,17896,12929,14408
5,source=AVISO_0.125_EAC_1999,27775,17897,18261,13100,14675
6,source=AVISO_0.125_EAC_2000,29658,18262,18627,14012,15646
7,source=AVISO_0.125_EAC_2001,28887,18628,18992,13633,15254
8,source=AVISO_0.125_EAC_2002,31589,18993,19357,14736,16853
9,source=AVISO_0.125_EAC_2003,31933,19358,19722,14300,17633


In [8]:
detection_files = sorted((config.output_root / "detections").glob("source=*.parquet"))

if detection_files:
    detection_counts = []
    for path in detection_files:
        frame = pd.read_parquet(path, columns=["Day", "Cyc"])
        detection_counts.append(
            {
                "partition": path.stem,
                "rows": len(frame),
                "first_day": frame.Day.min() if not frame.empty else pd.NA,
                "last_day": frame.Day.max() if not frame.empty else pd.NA,
                "AE": int(frame.Cyc.eq("AE").sum()),
                "CE": int(frame.Cyc.eq("CE").sum()),
            }
        )
    display(pd.DataFrame(detection_counts))
else:
    print(f"No detection files found in {config.output_root / 'detections'}")

,partition,rows,first_day,last_day,AE,CE
0,source=AVISO_0.125_EAC_1994,26312,16072,16435,12765,13547
1,source=AVISO_0.125_EAC_1995,28743,16436,16800,13479,15264
2,source=AVISO_0.125_EAC_1996,26198,16801,17166,11950,14248
3,source=AVISO_0.125_EAC_1997,29028,17167,17531,13699,15329
4,source=AVISO_0.125_EAC_1998,27337,17532,17896,12929,14408
5,source=AVISO_0.125_EAC_1999,27775,17897,18261,13100,14675
6,source=AVISO_0.125_EAC_2000,29658,18262,18627,14012,15646
7,source=AVISO_0.125_EAC_2001,28887,18628,18992,13633,15254
8,source=AVISO_0.125_EAC_2002,31589,18993,19357,14736,16853
9,source=AVISO_0.125_EAC_2003,31933,19358,19722,14300,17633


In [9]:
for name, path in {
    "tracked": config.output_root / "tracked" / "eddy_tracks.parquet",
    "processed": config.output_root / "processed" / "eddy_dataset_processed.parquet",
}.items():
    if path.exists():
        frame = pd.read_parquet(path)
        print(f"{name}: {len(frame):,} rows, {frame['Eddy'].nunique() if 'Eddy' in frame else 0:,} eddies")
        display(frame.head())
    else:
        print(f"Missing {name} output: {path}")

tracked: 705,392 rows, 47,902 eddies


,Day,Date,source_file,nxc,nyc,nCyc,nic,njc,xc,yc,...,Omega,Rc,psi0,R,err,Omega0_abs,Cyc,eddy_idx,Eddy,next_num
0,16072,1994-01-02,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...,696.0,27.0,AE,61,2,696.490264,27.808917,...,0.000002,90.469328,-9.033506,59.931186,0.945889,0.000003,AE,0,1,47903
1,16072,1994-01-02,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...,1516.0,40.0,AE,132,3,1515.427414,39.504351,...,0.000002,75.257967,-4.397429,53.729403,0.757313,0.000001,AE,1,2,47903
2,16072,1994-01-02,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...,1190.0,67.0,AE,103,5,1190.336549,67.117627,...,0.000001,138.718183,-14.065841,61.252108,0.356513,0.000002,AE,2,3,47903
3,16072,1994-01-02,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...,561.0,100.0,CE,49,7,560.752291,100.309167,...,-0.000008,80.757198,27.137091,53.098701,0.396162,0.000009,CE,3,4,47903
4,16072,1994-01-02,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...,1353.0,118.0,CE,118,9,1352.396540,118.385360,...,-0.000002,93.082713,8.759512,44.815789,0.716007,0.000002,CE,4,5,47903


processed: 453,309 rows, 8,979 eddies


,Eddy,Day,Date,Cyc,lon,lat,ic,jc,xc,yc,...,Omega,q11,q12,q22,Rc,psi0,AR,R,Age,source_file
0,1,16072,1994-01-02,AE,149.388784,-41.813765,35,9,397.941074,124.679076,...,0.000004,1.776830,-0.465120,0.687729,125.323701,-28.027626,1.942958,75.590039,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
1,1,16073,1994-01-03,AE,149.403113,-41.758355,35,9,399.259096,130.826827,...,0.000004,1.930643,-0.550539,0.684909,125.015516,-27.439127,2.118809,75.675331,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
2,1,16074,1994-01-04,AE,149.466532,-41.617063,35,11,405.092506,146.503209,...,0.000003,1.923138,-0.557851,0.691597,124.707331,-26.850629,2.118393,75.339928,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
3,1,16075,1994-01-05,AE,149.529951,-41.475772,36,12,410.925917,162.179590,...,0.000003,1.915632,-0.565163,0.698284,124.399146,-26.262130,2.118317,75.004526,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...
4,1,16076,1994-01-06,AE,149.576872,-41.354687,36,13,415.241842,175.614070,...,0.000003,1.808704,-0.508663,0.712941,124.090962,-25.673632,1.978232,74.989772,47,/srv/scratch/z5502183/AVISO_0.125/AVISO_0.125_...


## Run everything

This executes all four stages in order.

In [10]:
# run_all(config)